# Data Download and Prepare

This Notebook will download all the data used for this project. We will split the data into training and evaluation. So we can fine-tune the pretrained model and evaluate all changes in this research. 

## Data Sources

Since part of this and future research concentrate on energy forecast the energy domain is selected as our main focus. So, most data comes from the energy domain but some data is also selected from other areas like the M6, M5 and M4 competition datasets to get some benchmarks in other areas.

- Home Electricity, 15min, 100k timestamps, Home meter data from a single household
- OPSD Time Series (Europe), https://doi.org/10.25832/time_series/2019-06-05
- Renewable Energy and Electricity Demand (US), https://openenergyhub.ornl.gov/explore/dataset/renewable-energy-and-electricity-demand-time-series-dataset-with-exogenous-varia/information/?utm_source=chatgpt.com
- PJM Hourly Energy Consumption (US), https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption
  - Not used! Login required or API Key from Kaggle.
- M6, M5, M4 Competition Datasets, https://forecasters.org/resources/time-series-data/
  - Not used! Unclear data format for M4 and M5 competition. For M6 need to download and install yahoo API.
- FETS, Additional separate Dataset for covariate variables: https://zenodo.org/records/19418721

In [1]:
import polars as pl
import numpy as np
import niquests
from io import BytesIO
from pathlib import Path
from tqdm import tqdm

In [2]:
main_data_folder = Path("../data")
main_data_folder.mkdir(exist_ok=True)

In [3]:
def download_data(url: str, filepath: Path) -> None:
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with niquests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(filepath, "wb") as f, tqdm(
            desc=filepath.name,
            total=total,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))


def split_data(
    df: pl.DataFrame,
    time_col: str = "timestamp",
    train_ratio: float = 0.8,
    group_cols: list[str] | None = None,
    index_col: str = "series_index",
    index_start: int = 0,
) -> tuple[pl.DataFrame, pl.DataFrame, int]:
    
    if group_cols is None:
        df = df.with_columns(pl.lit(index_start).cast(pl.Int64).alias(index_col)).sort(time_col)
        n = len(df)
        train_df = df[: int(n * train_ratio)]
        test_df = df[int(n * train_ratio) :]
        max_group_index = df.select(pl.col(index_col).max()).item()
        return train_df, test_df, max_group_index

    # ensure sorted within each group
    df = df.sort(group_cols + [time_col])
    # add per-group index and group length
    df = df.with_columns([
        (pl.struct(group_cols)
         .rank("dense")
         .cast(pl.Int64) - 1 + index_start).alias(index_col),
        pl.int_range(0, pl.len()).over(group_cols).alias("group_index"),
        pl.len().over(group_cols).alias("group_size"),
    ])
    max_group_index = df.select(pl.col(index_col).max()).item()
    # compute split point (last X% = test)
    df = df.with_columns(
        (pl.col("group_size") * train_ratio).floor().cast(pl.Int64).alias("split_idx")
    )
    # split
    train_df = df.filter(pl.col("group_index") < pl.col("split_idx")).drop(["group_index", "group_size", "split_idx"])
    test_df  = df.filter(pl.col("group_index") >= pl.col("split_idx")).drop(["group_index", "group_size", "split_idx"])

    return train_df, test_df, max_group_index

# Home Electricity

This is a personal dataset from a single household with 80m².

In [128]:
# download data
data_url_home = "https://raw.githubusercontent.com/Richi0D/tirex_loss/refs/heads/main/data/home_electricity/home_electricity.csv"
data_path_home = main_data_folder / "home_electricity" / "home_electricity.csv"
download_data(data_url_home, data_path_home)

home_electricity.csv: 3.63MB [00:00, 55.2MB/s]                  


In [141]:
# split data
df_home_electricity = pl.read_csv(data_path_home, try_parse_dates=True)
df_home_electricity = df_home_electricity.rename({"Datum": "timestamp", "kW": "value"})
df_home_electricity = df_home_electricity.with_columns(pl.lit("home_electricity").alias("series_name"),
                                                       pl.col("timestamp").dt.replace_time_zone(None))
df_home_electricity_train, df_home_electricity_test, max_group_index = split_data(df_home_electricity)
print(f"Columns: {df_home_electricity_train.columns}")
print(f"Train Shape: {df_home_electricity_train.shape[0]}")
print(f"Test Shape: {df_home_electricity_test.shape[0]}")
df_home_electricity_train.head()

Columns: ['timestamp', 'value', 'series_name', 'series_index']
Train Shape: 80640
Test Shape: 20160


timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2023-06-10 22:00:00,0.308,"""home_electricity""",0
2023-06-10 22:15:00,0.14,"""home_electricity""",0
2023-06-10 22:30:00,0.14,"""home_electricity""",0
2023-06-10 22:45:00,0.14,"""home_electricity""",0
2023-06-10 23:00:00,0.148,"""home_electricity""",0


# OPSD

Load, wind, solar and prices from 32 European countries.
15min and 60min stacked dataset is used.

In [123]:
# download data
data_url_opsd_15m = "https://data.open-power-system-data.org/time_series/latest/time_series_15min_stacked.csv"
data_url_opsd_60m = "https://data.open-power-system-data.org/time_series/latest/time_series_60min_stacked.csv"
data_path_opsd_15m = main_data_folder / "opsd" / "opsd_15min_stacked.csv"
data_path_opsd_60m = main_data_folder / "opsd" / "opsd_60min_stacked.csv"
download_data(data_url_opsd_15m, data_path_opsd_15m)
download_data(data_url_opsd_60m, data_path_opsd_60m)

opsd_15min_stacked.csv: 669MB [00:09, 71.9MB/s] 
opsd_60min_stacked.csv: 841MB [00:17, 49.9MB/s] 


In [142]:
# split data
df_opsd_15m = pl.read_csv(data_path_opsd_15m, try_parse_dates=True)
df_opsd_15m = df_opsd_15m.rename({"utc_timestamp": "timestamp", "data": "value"})
df_opsd_15m = df_opsd_15m.with_columns(
    pl.concat_str([pl.col("region"), pl.col("variable"), pl.col("attribute"), pl.lit("15")], separator="_").alias("series_name"),
    pl.col("timestamp").dt.replace_time_zone(None)
).drop(["region", "variable", "attribute"])
df_opsd_15m_train, df_opsd_15m_test, max_group_index = split_data(df_opsd_15m,
                                                group_cols=["series_name"],
                                                index_start=max_group_index + 1)
print(f"Columns: {df_opsd_15m_train.columns}")
print(f"Train Shape: {df_opsd_15m_train.shape[0]}")
print(f"Test Shape: {df_opsd_15m_test.shape[0]}")
df_opsd_15m_train.head()

Columns: ['timestamp', 'value', 'series_name', 'series_index']
Train Shape: 8479014
Test Shape: 2119784


timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2015-01-01 00:15:00,5966.8,"""AT_load_actual_entsoe_transpar…",1
2015-01-01 00:30:00,5935.6,"""AT_load_actual_entsoe_transpar…",1
2015-01-01 00:45:00,5934.4,"""AT_load_actual_entsoe_transpar…",1
2015-01-01 01:00:00,5750.8,"""AT_load_actual_entsoe_transpar…",1
2015-01-01 01:15:00,5777.6,"""AT_load_actual_entsoe_transpar…",1


In [143]:
# split data
df_opsd_60m = pl.read_csv(data_path_opsd_60m, try_parse_dates=True)
df_opsd_60m = df_opsd_60m.rename({"utc_timestamp": "timestamp", "data": "value"})
df_opsd_60m = df_opsd_60m.with_columns(
    pl.concat_str([pl.col("region"), pl.col("variable"), pl.col("attribute"), pl.lit("60")], separator="_").alias("series_name"),
    pl.col("timestamp").dt.replace_time_zone(None)
).drop(["region", "variable", "attribute"])
df_opsd_60m_train, df_opsd_60m_test, max_group_index = split_data(df_opsd_60m,
                                                group_cols=["series_name"],
                                                index_start=max_group_index + 1)
print(f"Columns: {df_opsd_60m_train.columns}")
print(f"Train Shape: {df_opsd_60m_train.shape[0]}")
print(f"Test Shape: {df_opsd_60m_test.shape[0]}")
df_opsd_60m_train.head()

Columns: ['timestamp', 'value', 'series_name', 'series_index']
Train Shape: 10949008
Test Shape: 2737395


timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2015-01-01 00:00:00,5946.0,"""AT_load_actual_entsoe_transpar…",60
2015-01-01 01:00:00,5726.0,"""AT_load_actual_entsoe_transpar…",60
2015-01-01 02:00:00,5347.0,"""AT_load_actual_entsoe_transpar…",60
2015-01-01 03:00:00,5249.0,"""AT_load_actual_entsoe_transpar…",60
2015-01-01 04:00:00,5309.0,"""AT_load_actual_entsoe_transpar…",60


# Open Energy Hub

Renewable Energy and Electricity Demand Time Series Dataset with Exogenous Variables at 5-minute Interval.

The database contains 12 columns:
- date
- station (1: Winter, 2: Spring, 3: Summer, 4: Autumn)
- day of the week (0: Monday, ... , 6: Sunday)
- DHI (W/m2)
- DNI (W/m2)
- GHI (W/m2)
- wind speed (m/s)
- humidity (%)
- temperature (degrees)
- solar energy production (MW)
- wind energy production (MW)
- electricity demand (MW)

We use only the last 3 columns which provide data about production and demand.

In [132]:
# download data
data_url_caiso = "https://data.mendeley.com/public-files/datasets/fdfftr3tc2/files/fff037a3-d0e4-496f-92f7-5c5820a734f1/file_downloaded"
data_path_caiso = main_data_folder / "openenergyhub" / "openenergyhub.csv"
download_data(data_url_caiso, data_path_caiso)

openenergyhub.csv: 100%|██████████| 28.4M/28.4M [00:03<00:00, 7.45MB/s]


In [145]:
# split data
df_caiso = pl.read_csv(data_path_caiso, try_parse_dates=True)
df_caiso = df_caiso.rename({"Time": "timestamp",
                            "PV_production": "caiso_pv_production",
                            "Wind_production": "caiso_wind_production",
                            "Electric_demand": "caiso_electric_demand"})
df_caiso = df_caiso.with_columns(
    pl.col("timestamp").str.strptime(pl.Datetime, format="%Y-%m-%d-T%H:%M", strict=False),
    )
df_caiso_melt = df_caiso.unpivot(index="timestamp",
                                on=["caiso_pv_production", "caiso_wind_production", "caiso_electric_demand"],
                                variable_name="series_name",
                                value_name="value")
df_caiso_melt = df_caiso_melt.with_columns(pl.col('value').cast(pl.Float64))

df_caiso_train, df_caiso_test, max_group_index = split_data(df_caiso_melt,
                                                             group_cols=["series_name"],
                                                               index_start=max_group_index + 1)
print(f"Columns: {df_caiso_train.columns}")
print(f"Train Shape: {df_caiso_train.shape[0]}")
print(f"Test Shape: {df_caiso_test.shape[0]}")
df_caiso_train.head()

Columns: ['timestamp', 'series_name', 'value', 'series_index']
Train Shape: 757554
Test Shape: 189390


timestamp,series_name,value,series_index
datetime[μs],str,f64,i64
2019-01-01 00:00:00,"""caiso_electric_demand""",22216.0,358
2019-01-01 00:05:00,"""caiso_electric_demand""",22106.0,358
2019-01-01 00:10:00,"""caiso_electric_demand""",22130.0,358
2019-01-01 00:15:00,"""caiso_electric_demand""",22040.0,358
2019-01-01 00:20:00,"""caiso_electric_demand""",21963.0,358


# PJM

Contains Hourly Energy Consumption data from the US.
Only American Electric Power Dataset is used here. 

In [ ]:
raise NotImplementedError("PJM data download not implemented yet. Kaggle implementation is needed.")

# download data
data_url_pjm = ""
data_path_pjm = main_data_folder / "pjm" / "pjm_aep_hourly.csv"
download_data(data_url_pjm, data_path_pjm)

# IIF

M6, M5, M4 Competition Datasets

In [118]:
# download data
data_url_iif_m4_hourly_train = "https://github.com/Mcompetitions/M4-methods/raw/refs/heads/master/Dataset/Train/Hourly-train.csv"
data_path_iif_m4_hourly_train = main_data_folder / "iif" / "iif_m4_hourly_train.csv"
download_data(data_url_iif_m4_hourly_train, data_path_iif_m4_hourly_train)

data_url_iif_m4_daily_train = "https://github.com/Mcompetitions/M4-methods/raw/refs/heads/master/Dataset/Train/Daily-train.csv"
data_path_iif_m4_daily_train = main_data_folder / "iif" / "iif_m4_daily_train.csv"
download_data(data_url_iif_m4_daily_train, data_path_iif_m4_daily_train)

data_url_iif_m4_monthly_train = "https://github.com/Mcompetitions/M4-methods/raw/refs/heads/master/Dataset/Train/Monthly-train.csv"
data_path_iif_m4_monthly_train = main_data_folder / "iif" / "iif_m4_monthly_train.csv"
download_data(data_url_iif_m4_monthly_train, data_path_iif_m4_monthly_train)

data_url_iif_m4_yearly_train = "https://github.com/Mcompetitions/M4-methods/raw/refs/heads/master/Dataset/Train/Yearly-train.csv"
data_path_iif_m4_yearly_train = main_data_folder / "iif" / "iif_m4_yearly_train.csv"
download_data(data_url_iif_m4_yearly_train, data_path_iif_m4_yearly_train)

# test data
data_url_iif_m4_hourly_test = "https://github.com/Mcompetitions/M4-methods/raw/refs/heads/master/Dataset/Test/Hourly-test.csv"
data_path_iif_m4_hourly_test = main_data_folder / "iif" / "iif_m4_hourly_test.csv"
download_data(data_url_iif_m4_hourly_test, data_path_iif_m4_hourly_test)

data_url_iif_m4_daily_test = "https://github.com/Mcompetitions/M4-methods/raw/refs/heads/master/Dataset/Test/Daily-test.csv"
data_path_iif_m4_daily_test = main_data_folder / "iif" / "iif_m4_daily_test.csv"
download_data(data_url_iif_m4_daily_test, data_path_iif_m4_daily_test)

data_url_iif_m4_monthly_test = "https://github.com/Mcompetitions/M4-methods/raw/refs/heads/master/Dataset/Test/Monthly-test.csv"
data_path_iif_m4_monthly_test = main_data_folder / "iif" / "iif_m4_monthly_test.csv"
download_data(data_url_iif_m4_monthly_test, data_path_iif_m4_monthly_test)

data_url_iif_m4_yearly_test = "https://github.com/Mcompetitions/M4-methods/raw/refs/heads/master/Dataset/Test/Yearly-test.csv"
data_path_iif_m4_yearly_test = main_data_folder / "iif" / "iif_m4_yearly_test.csv"
download_data(data_url_iif_m4_yearly_test, data_path_iif_m4_yearly_test)

iif_m4_hourly_train.csv: 2.24MB [00:00, 28.1MB/s]                  
iif_m4_daily_train.csv: 91.3MB [00:02, 33.1MB/s]                            
iif_m4_monthly_train.csv: 87.4MB [00:05, 18.2MB/s]                            
iif_m4_yearly_train.csv: 24.2MB [00:00, 62.0MB/s]                   
iif_m4_hourly_test.csv: 130kB [00:00, 33.2MB/s]                    
iif_m4_daily_test.csv: 563kB [00:00, 23.1MB/s]                   
iif_m4_monthly_test.csv: 7.57MB [00:00, 29.1MB/s]                            
iif_m4_yearly_test.csv: 1.42MB [00:00, 22.2MB/s]                  


In [121]:
df_iif_m4_hourly_train = pl.read_csv(data_path_iif_m4_hourly_train, try_parse_dates=True, ignore_errors=True)
df_iif_m4_hourly_train

V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,…,V925,V926,V927,V928,V929,V930,V931,V932,V933,V934,V935,V936,V937,V938,V939,V940,V941,V942,V943,V944,V945,V946,V947,V948,V949,V950,V951,V952,V953,V954,V955,V956,V957,V958,V959,V960,V961
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""H1""",605,586,586,559,511,443,422,395,382,370,383,397,420,455,493,554,610,666,715,755,778,794,806,808,776,723,709,660,585,527,462,437,413,407,404,420,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""H2""",3124,2990,2862,2809,2544,2201,1996,1861,1735,1713,1724,1798,1891,2037,2102,2163,2269,2404,2515,2621,2745,2816,2938,3022,2976,2892,2784,2725,2530,2211,1995,1833,1768,1712,1707,1762,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""H3""",1828,1806,1897,1750,1679,1620,1463,1342,1192,1108,1058,1024,1031,1091,1208,1337,1435,1515,1593,1667,1753,1768,1823,1813,1842,1838,1800,1761,1670,1609,1467,1309,1189,1102,1054,1017,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""H4""",6454,6324,6075,5949,5858,5579,5163,4790,4478,4227,4016,3879,3821,3823,3960,4169,4377,4597,4864,5183,5440,5707,5937,6089,6097,6018,5783,5655,5581,5320,4909,4509,4189,3964,3794,3680,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""H5""",4263,4297,4236,4080,3883,3672,3248,2841,2513,2275,2104,1988,1958,2006,2076,2209,2372,2599,2880,3127,3297,3506,3667,3752,3842,3851,3728,3578,3421,3252,2910,2559,2293,2104,1974,1893,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""H410""",153,196,163,131,82,53,57,54,46,72,97,135,121,166,175,146,165,140,148,114,125,107,138,95,64,59,52,34,44,34,62,61,88,76,109,153,…,"""125""","""109""","""137""","""137""","""147""","""208""","""234""","""137""","""114""","""90""","""84""","""104""","""83""","""46""","""34""","""38""","""22""","""26""","""36""","""104""","""173""","""174""","""116""","""106""","""112""","""103""","""104""","""132""","""165""","""191""","""186""","""119""","""108""","""70""","""72""","""79""","""77"""
"""H411""",24,30,22,14,19,20,24,45,38,60,67,87,108,103,91,102,81,87,78,70,56,44,37,40,26,18,22,16,19,38,57,84,65,81,92,140,…,"""115""","""123""","""128""","""137""","""181""","""166""","""145""","""96""","""81""","""74""","""55""","""45""","""51""","""31""","""27""","""21""","""29""","""40""","""69""","""169""","""258""","""214""","""150""","""131""","""126""","""121""","""139""","""122""","""195""","""166""","""154""","""132""","""76""","""66""","""67""","""51""","""42"""
"""H412""",19,30,12,16,13,15,15,21,21,29,35,55,66,65,69,62,73,64,63,55,45,32,29,28,22,15,14,17,14,13,18,41,46,37,52,54,…,"""71""","""80""","""92""","""100""","""159""","""265""","""290""","""127""","""92""","""73""","""70""","""35""","""27""","""23""","""19""","""17""","""23""","""19""","""18""","""42""","""90""","""112""","""86""","""79""","""68""","""71""","""75""","""111""","""166""","""225""","""278""

# Data Join

In [148]:
columns_order = ["timestamp", "value", "series_name", "series_index"]

df_to_join_train = [df_home_electricity_train.select(columns_order),
            df_opsd_15m_train.select(columns_order),
            df_opsd_60m_train.select(columns_order),
            df_caiso_train.select(columns_order),
            ]
df_joined_train = pl.concat(df_to_join_train, how="vertical")
df_joined_train

timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2023-06-10 22:00:00,0.308,"""home_electricity""",0
2023-06-10 22:15:00,0.14,"""home_electricity""",0
2023-06-10 22:30:00,0.14,"""home_electricity""",0
2023-06-10 22:45:00,0.14,"""home_electricity""",0
2023-06-10 23:00:00,0.148,"""home_electricity""",0
…,…,…,…
2021-05-26 18:45:00,4132.0,"""caiso_wind_production""",360
2021-05-26 18:50:00,4168.0,"""caiso_wind_production""",360
2021-05-26 18:55:00,4196.0,"""caiso_wind_production""",360


In [149]:
df_to_join_test = [df_home_electricity_test.select(columns_order),
            df_opsd_15m_test.select(columns_order),
            df_opsd_60m_test.select(columns_order),
            df_caiso_test.select(columns_order),
            ]
df_joined_test = pl.concat(df_to_join_test, how="vertical")
df_joined_test

timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2025-09-27 22:00:00,0.564,"""home_electricity""",0
2025-09-27 22:15:00,0.564,"""home_electricity""",0
2025-09-27 22:30:00,0.556,"""home_electricity""",0
2025-09-27 22:45:00,0.568,"""home_electricity""",0
2025-09-27 23:00:00,0.56,"""home_electricity""",0
…,…,…,…
2021-12-31 23:35:00,3778.0,"""caiso_wind_production""",360
2021-12-31 23:40:00,3755.0,"""caiso_wind_production""",360
2021-12-31 23:45:00,3751.0,"""caiso_wind_production""",360


In [153]:
df_sequences = df_joined_train.select(["series_name", "series_index"]).unique().sort("series_index")
df_sequences

series_name,series_index
str,i64
"""home_electricity""",0
"""AT_load_actual_entsoe_transpar…",1
"""AT_load_forecast_entsoe_transp…",2
"""AT_price_day_ahead_15""",3
"""AT_solar_generation_actual_15""",4
…,…
"""UA_load_actual_entsoe_transpar…",356
"""UA_load_forecast_entsoe_transp…",357
"""caiso_electric_demand""",358


In [154]:
# save datasets
df_joined_train.write_csv(main_data_folder / "train.csv")
df_joined_test.write_csv(main_data_folder / "test.csv")
df_sequences.write_csv(main_data_folder / "description_sequences.csv")

# FETS

Separate Dataset used for fine tuning and covariate tests.

In [ ]:
# download data
data_url_fets = "https://zenodo.org/records/19418721/files/FETS_Dataset.zip?download=1"
data_path_fets = main_data_folder / "fets" / "FETS_Dataset.zip"
download_data(data_url_fets, data_path_fets)

FETS_Dataset.zip: 100%|██████████| 383M/383M [00:12<00:00, 32.0MB/s] 


In [30]:
# unzip data
import zipfile
with zipfile.ZipFile(data_path_fets, 'r') as zip_ref:
    zip_ref.extractall(main_data_folder / "fets")

In [ ]:
# split data
path_de_power = (main_data_folder / "fets" / "sf_storage" / 
                 "Workspace" / "Marco" / "0_data" / "3_published_data" / 
                 "Load" / "DE Power - Load DE"
                )
path_de_power_load = path_de_power / "public_power.parquet"
path_de_power_covariates_weather = path_de_power / "covariates" / "weatherDE_grouped3_agg.parquet"
path_de_power_covariates_calendar = path_de_power / "covariates" / "calendar_DE_2010_2030.parquet"

In [9]:
df_fets_load = pl.read_parquet(path_de_power_load)
df_fets_load.head()

timestamp,Hydro pumped storage consumption,Cross border electricity trading,Nuclear,Hydro Run-of-River,Biomass,Fossil brown coal / lignite,Fossil hard coal,Fossil oil,Fossil gas,Geothermal,Hydro water reservoir,Hydro pumped storage,Others,Waste,Wind offshore,Wind onshore,Solar,Load,Residual load,Renewable share of load,Renewable share of generation,Fossil coal-derived gas
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2019-12-31 23:00:00 UTC,-194.9,680.3,8095.9,1558.7,4935.2,9203.9,2104.9,453.7,5951.9,27.9,451.2,1026.3,467.1,1399.9,501.3,5909.1,0.0,43881.8,37471.3,32.0,33.3,null
2019-12-31 23:15:00 UTC,-225.1,680.3,8082.5,1553.7,4929.0,9227.9,2065.5,453.7,5826.5,27.9,385.7,628.8,461.2,1409.5,501.7,5905.6,0.0,43639.6,37232.3,32.0,33.7,null
2019-12-31 23:30:00 UTC,-286.2,680.3,8082.2,1548.2,4912.7,9259.8,2059.5,453.8,5737.6,27.9,274.5,528.6,461.0,1416.4,540.7,6108.3,0.0,43330.9,36681.9,32.5,34.0,null
2019-12-31 23:45:00 UTC,-397.0,680.3,8077.4,1548.2,4912.8,9272.1,2091.7,453.7,5666.0,27.8,98.7,312.0,461.1,1408.1,655.5,6240.5,0.0,43149.5,36253.5,32.8,34.3,null
2020-01-01 00:00:00 UTC,-285.0,1034.6,8090.7,1548.7,4910.1,9331.5,2052.5,453.7,5478.3,27.9,244.1,723.0,461.1,1385.0,890.6,6094.6,0.0,43017.3,36032.1,33.4,34.4,null


In [11]:
df_fets_weather = pl.read_parquet(path_de_power_covariates_weather)
df_fets_weather.head()

timestamp,DE_group0_temperature_2m,DE_group0_shortwave_radiation_instant,DE_group0_direct_radiation_instant,DE_group0_wind_speed_80m,DE_group0_wind_speed_120m,DE_group0_wind_speed_180m,DE_group1_temperature_2m,DE_group1_shortwave_radiation_instant,DE_group1_direct_radiation_instant,DE_group1_wind_speed_80m,DE_group1_wind_speed_120m,DE_group1_wind_speed_180m,DE_group2_temperature_2m,DE_group2_shortwave_radiation_instant,DE_group2_direct_radiation_instant,DE_group2_wind_speed_80m,DE_group2_wind_speed_120m,DE_group2_wind_speed_180m
"datetime[ns, UTC]",f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
2022-01-01 00:00:00 UTC,10.325175,0.0,0.0,32.054855,34.405029,null,9.012249,0.0,0.0,22.668848,25.114269,null,9.854263,0.0,0.0,29.343624,31.966272,null
2022-01-01 01:00:00 UTC,10.276201,0.0,0.0,31.290421,33.58754,null,8.944674,0.0,0.0,22.577488,24.278522,null,9.919207,0.0,0.0,29.197962,31.759922,null
2022-01-01 02:00:00 UTC,10.24859,0.0,0.0,30.553743,32.782143,null,8.772588,0.0,0.0,21.097713,23.046619,null,9.946275,0.0,0.0,28.40369,30.992371,null
2022-01-01 03:00:00 UTC,10.199896,0.0,0.0,29.543827,31.858114,null,8.649749,0.0,0.0,19.513308,22.139465,null,9.90323,0.0,0.0,27.24144,29.89505,null
2022-01-01 04:00:00 UTC,10.088141,0.0,0.0,28.185009,30.542042,null,8.438283,0.0,0.0,18.207626,20.642775,null,9.820805,0.0,0.0,26.090105,28.669556,null


In [12]:
df_fets_calendar = pl.read_parquet(path_de_power_covariates_calendar)
df_fets_calendar.head()

timestamp,hour_sin,hour_cos,minute_sin,minute_cos,weekday_sin,weekday_cos,day_sin,day_cos,month_sin,month_cos,is_holiday,is_weekend
datetime[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8
2010-01-01 00:00:00,0.0,1.0,0.0,1.0,-0.974928,-0.222521,0.201299,0.97953,0.5,0.866025,1,1
2010-01-01 00:15:00,0.0,1.0,1.0,2.8328e-16,-0.974928,-0.222521,0.201299,0.97953,0.5,0.866025,1,1
2010-01-01 00:30:00,0.0,1.0,5.6655e-16,-1.0,-0.974928,-0.222521,0.201299,0.97953,0.5,0.866025,1,1
2010-01-01 00:45:00,0.0,1.0,-1.0,-1.8370e-16,-0.974928,-0.222521,0.201299,0.97953,0.5,0.866025,1,1
2010-01-01 01:00:00,0.258819,0.965926,0.0,1.0,-0.974928,-0.222521,0.201299,0.97953,0.5,0.866025,1,1


In [15]:
# just select load data to fine tune on it
variable = "Load"
df_fets = df_fets_load.select(
    ["timestamp", variable]).rename(
        {variable: "value"}).with_columns(
            pl.lit(0).alias("series_index"))

df_fets.write_csv(main_data_folder / "data_fets.csv")
df_fets

timestamp,value,series_index
"datetime[ns, UTC]",f64,i32
2019-12-31 23:00:00 UTC,43881.8,0
2019-12-31 23:15:00 UTC,43639.6,0
2019-12-31 23:30:00 UTC,43330.9,0
2019-12-31 23:45:00 UTC,43149.5,0
2020-01-01 00:00:00 UTC,43017.3,0
…,…,…
2025-10-13 20:45:00 UTC,51524.0,0
2025-10-13 21:00:00 UTC,50427.2,0
2025-10-13 21:15:00 UTC,49845.8,0
